In [1]:
! nvidia-smi

Fri Apr  3 18:31:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 5000 Blac...    On  |   00000000:E1:00.0 Off |                  Off |
| 38%   35C    P8              6W /  300W |       0MiB /  48935MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Install required packages

In [2]:
! pip show torch torchaudio torchcodec

Name: torch
Version: 2.8.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /venv/main/lib/python3.11/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: fairseq2n, omnilingual-asr, torchaudio, torchvision
---
Name: torchaudio
Version: 2.8.0+cu128
Summary: An audio package for PyTorch
Home-page: https://github.com/pytorch/audio
Author: Soumith Chintala, David Pollack, Sean Naren, Peter Goldsborough, Moto Hira, Caroline Chen, Jeff Hwang, Zhaoheng Ni, Xiaohui Zhang
Author-email: soum

In [3]:
# ! pip uninstall torch torchvision torchaudio -y
# ! pip install torch=="2.8.0" torchvision=="0.23.0" torchaudio=="2.8.0" torchcodec=="0.7" --index-url https://download.pytorch.org/whl/cu128 -q

In [4]:
! pip install "transformers<5.0.0" -q
# ! pip install datacollective -q
! conda install -c conda-forge libsndfile==1.0.31 -y -q
! pip install git+https://github.com/facebookresearch/omnilingual-asr.git -q
! pip install Unidecode -q
! pip install evaluate -q
! pip install jiwer -q
! pip install datasets=="4.8.4" -q
! pip install librosa -q

Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

# All requested packages already installed.



# Imports

In [5]:
! wget -O text_tools.py https://raw.githubusercontent.com/facebookresearch/omnilingual-asr/refs/heads/main/workflows/dataprep/text_tools.py --no-clobber
! wget -O punctuations.lst https://raw.githubusercontent.com/facebookresearch/omnilingual-asr/refs/heads/main/workflows/dataprep/punctuations.lst --no-clobber
! wget -O norm_config_module.py https://raw.githubusercontent.com/facebookresearch/omnilingual-asr/refs/heads/main/workflows/dataprep/norm_config_module.py --no-clobber

File ‘text_tools.py’ already there; not retrieving.
File ‘punctuations.lst’ already there; not retrieving.
File ‘norm_config_module.py’ already there; not retrieving.


In [6]:
# from datacollective import download_dataset, get_dataset_details
# from datacollective import load_dataset as dc_load_dataset
from datasets import load_dataset, get_dataset_config_info, Value, Dataset, Audio
from text_tools import text_normalize
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from transformers import Wav2Vec2ForCTC, AutoProcessor, pipeline, logging
import torch
from tqdm import tqdm
import librosa
import soundfile as sf
import io
from IPython.display import Audio as AudioDisplay
import tarfile
import os
import pandas as pd
import numpy as np
import json
import evaluate
import gc
# from kaggle_secrets import UserSecretsClient

# Load Data

In [7]:
data = {}
long_data = {}

## Omnilingual ASR Corpus
Covered languages: Nigerian Fulfulde (fuv), Central-Eastern Niger Fulfulde (fuq), Soninke (snk).

In [8]:
omni_fuv_test = load_dataset("facebook/omnilingual-asr-corpus", "fuv_Latn", split="test")
omni_fuq_test = load_dataset("facebook/omnilingual-asr-corpus", "fuq_Latn", split="test")
omni_snk_test = load_dataset("facebook/omnilingual-asr-corpus", "snk_Latn", split="test")

# Rename "raw_text" to "transcription" for consistency
omni_fuv_test = omni_fuv_test.rename_columns({"raw_text": "transcription"})
omni_fuq_test = omni_fuq_test.rename_columns({"raw_text": "transcription"})
omni_snk_test = omni_snk_test.rename_columns({"raw_text": "transcription"})

# Resample audio to 16kHz
omni_fuv_test = omni_fuv_test.cast_column("audio", Audio(sampling_rate=16000))
omni_fuq_test = omni_fuq_test.cast_column("audio", Audio(sampling_rate=16000))
omni_snk_test = omni_snk_test.cast_column("audio", Audio(sampling_rate=16000))

Resolving data files:   0%|          | 0/851 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/311 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/309 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/851 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/311 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/309 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/851 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/311 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/309 [00:00<?, ?it/s]

In [9]:
# Fix the samples OmniASR fails on by adding one second of silence at the end of the audio
fuv_idx = [26, 37, 65]
fuq_idx = [188]

omni_fuv_test = omni_fuv_test.cast_column("audio", Audio(decode=False))
omni_fuq_test = omni_fuq_test.cast_column("audio", Audio(decode=False))

# Store the original audio bytes for the affected samples
# original_audio_bytes_fuv = {idx: omni_fuv_test[idx]['audio']['bytes'] for idx in fuv_idx}
# original_audio_bytes_fuq = {idx: omni_fuq_test[idx]['audio']['bytes'] for idx in fuq_idx}

def fix_audio(sample, idx):
    if (sample['language'] == 'fuv_Latn' and idx in fuv_idx) or (sample['language'] == 'fuq_Latn' and idx in fuq_idx):
        audio_bytes = sample['audio']['bytes']
        path = sample['audio']['path']
        # Load the audio from bytes
        input_stream = io.BytesIO(audio_bytes)
        with sf.SoundFile(input_stream) as f:
            original_format = f.format
            original_subtype = f.subtype
            original_sr = f.samplerate
            original_channels = f.channels
        print(f"Original format: {original_format}, subtype: {original_subtype}, sample rate: {original_sr}, channels: {original_channels}")
        input_stream.seek(0)  # Reset the stream position to the beginning
        audio, sr = librosa.load(input_stream, sr=original_sr)
        # Add one second of silence at the end
        silence = np.zeros(sr, dtype=np.float32)
        padded_waveform = np.concatenate((audio, silence))
        output_buffer = io.BytesIO()
        sf.write(output_buffer, padded_waveform, sr, format=original_format, subtype=original_subtype)
        padded_bytes = output_buffer.getvalue()
        # Update the sample
        sample['audio'] = {
            'bytes': padded_bytes,
            'path': path
        }
    return sample

"""
def restore_original_audio(sample):
    index = sample['idx']
    if sample['language'] == 'fuv_Latn' and index in fuv_idx:
        sample['audio'] = {
            'bytes': original_audio_bytes_fuv[index],
            'path': sample['audio']['path']
        }
    elif sample['language'] == 'fuq_Latn' and index in fuq_idx:
        sample['audio'] = {
            'bytes': original_audio_bytes_fuq[index],
            'path': sample['audio']['path']
        }
    return sample
"""

omni_fuv_test = omni_fuv_test.map(fix_audio, with_indices=True)
omni_fuq_test = omni_fuq_test.map(fix_audio, with_indices=True)

omni_fuv_test = omni_fuv_test.cast_column("audio", Audio(decode=True, sampling_rate=16000))
omni_fuq_test = omni_fuq_test.cast_column("audio", Audio(decode=True, sampling_rate=16000))

for idx in fuv_idx:
    print(f"Original duration: {omni_fuv_test[idx]['duration']} seconds. New duration: {len(omni_fuv_test[idx]['audio']['array']) / omni_fuv_test[idx]['audio']['sampling_rate']} seconds.")
for idx in fuq_idx:
    print(f"Original duration: {omni_fuq_test[idx]['duration']} seconds. New duration: {len(omni_fuq_test[idx]['audio']['array']) / omni_fuq_test[idx]['audio']['sampling_rate']} seconds.")

# def create_index(sample, idx):
#     sample['idx'] = idx
#     return sample

# omni_fuv_test = omni_fuv_test.map(create_index, with_indices=True)
# omni_fuq_test = omni_fuq_test.map(create_index, with_indices=True)

Original duration: 75.0185260770975 seconds. New duration: 76.0185625 seconds.
Original duration: 90.02378684807256 seconds. New duration: 91.0238125 seconds.
Original duration: 90.01437641723356 seconds. New duration: 91.014375 seconds.
Original duration: 15.007891156462586 seconds. New duration: 16.007875 seconds.


In [10]:
# Sort the dataset by duration
omni_fuv_test = omni_fuv_test.sort("duration", reverse=True)
omni_fuq_test = omni_fuq_test.sort("duration", reverse=True)
omni_snk_test = omni_snk_test.sort("duration", reverse=True)

# Filter out long samples for fuq_Latn
# def is_short(sample):
#     return sample['duration'] < 40

# omni_fuq_test = omni_fuq_test.filter(is_short)

omni_fuq_test_df = omni_fuq_test.to_pandas()

omni_fuq_test_df = omni_fuq_test_df[omni_fuq_test_df['duration'] < 40]

# Convert back to Hugging Face Dataset
omni_fuq_test = Dataset.from_pandas(omni_fuq_test_df)

omni_fuq_test = omni_fuq_test.cast_column("audio", Audio(sampling_rate=16000))

In [11]:
long_data["fuv_Latn"] = [{"Omni_Corpus": omni_fuv_test}]
data["fuq_Latn"] = [{"Omni_Corpus": omni_fuq_test}]
long_data["snk_Latn"] = [{"Omni_Corpus": omni_snk_test}]

## FLEURS
Covered languages: Nigerian Fulfulde (fuv).

In [12]:
fleurs_fuv_test = load_dataset("google/fleurs", "ff_sn", split="test", revision="refs/pr/31")

# Resample audio to 16kHz
fleurs_fuv_test = fleurs_fuv_test.cast_column("audio", Audio(sampling_rate=16000))

In [13]:
def calculate_duration(sample):
    duration = len(sample['audio']['array']) / sample['audio']['sampling_rate']
    sample['duration'] = duration
    return sample

# Calculate duration for each audio sample and add it as a new column
fleurs_fuv_test = fleurs_fuv_test.map(calculate_duration)

# Sort the dataset by duration
fleurs_fuv_test = fleurs_fuv_test.sort("duration", reverse=True)

data["fuv_Latn"] = [{"Fleurs": fleurs_fuv_test}]

## WaxalNLP
Covered languages: Ewe (ewe), Dagbani (dag).

In [14]:
info = get_dataset_config_info("google/WaxalNLP", "ewe_asr")
ewe_latn_waxal_features = info.features.copy()

ewe_latn_waxal_features["__index_level_0__"] = Value("int64")
ewe_latn_waxal_features["audio"] = Audio(sampling_rate=16000)

waxal_ewe_test = load_dataset("google/WaxalNLP", "ewe_asr", split="test", streaming=True, features=ewe_latn_waxal_features)

# Calculate duration for each audio sample and add it as a new column
waxal_ewe_test = waxal_ewe_test.map(calculate_duration)
ewe_latn_waxal_features["duration"] = Value("float64")

# Filter out long samples
def is_short(sample):
    return sample['duration'] < 40

waxal_ewe_test = waxal_ewe_test.filter(is_short)

# Drop samples with None transcriptions
def drop_none_transcriptions(sample):
    return sample['transcription'] is not None

waxal_ewe_test = waxal_ewe_test.filter(drop_none_transcriptions)

waxal_ewe_test = waxal_ewe_test.cast(ewe_latn_waxal_features)

data["ewe_Latn"] = [{"Waxal": waxal_ewe_test}]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/106 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/106 [00:00<?, ?it/s]

In [15]:
info = get_dataset_config_info("google/WaxalNLP", "dag_asr")
dag_latn_waxal_features = info.features.copy()

dag_latn_waxal_features["__index_level_0__"] = Value("int64")
dag_latn_waxal_features["audio"] = Audio(sampling_rate=16000)

waxal_dag_test = load_dataset("google/WaxalNLP", "dag_asr", split="test", streaming=True, features=dag_latn_waxal_features)

# Calculate duration for each audio sample and add it as a new column
waxal_dag_test = waxal_dag_test.map(calculate_duration)
dag_latn_waxal_features["duration"] = Value("float64")

# Filter out long samples
waxal_dag_test = waxal_dag_test.filter(is_short)

waxal_dag_test = waxal_dag_test.cast(dag_latn_waxal_features)

data["dag_Latn"] = [{"Waxal": waxal_dag_test}]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

## Moore Speech Corpora
Covered languages: Mooré (mos).

In [16]:
# # Kaggle Secrets Client to manage API keys securely in Kaggle notebooks
# user_secrets = UserSecretsClient()

# # Set your HF token as an environment variable
# os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [17]:
mos_corp_test = load_dataset("goaicorp/GOAI-MooreSpeechCorpora", split="test")

# Resample audio to 16kHz
mos_corp_test = mos_corp_test.cast_column("audio", Audio(sampling_rate=16000))

# Rename "text" to "transcription" for consistency
mos_corp_test = mos_corp_test.rename_columns({"text": "transcription"})

# Sort the dataset by duration
mos_corp_test = mos_corp_test.sort("duration", reverse=True)

README.md: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4523 [00:00<?, ? examples/s]

Generating full split:   0%|          | 0/5535 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1012 [00:00<?, ? examples/s]

In [18]:
data['mos_Latn'] = [{"Moore_Speech_Corpora": mos_corp_test}]

## Common Voice
Covered languages: Dagbani (dag).

The dataset can be downloaded using the Mozilla Data Collective API python client.

In [19]:
# DATASET_ID = "cmn2cy2su01iymm07xfr6ul2b"

In [20]:
# dataset_path = download_dataset(DATASET_ID)
# dataset_path

In [21]:
# details = get_dataset_details(DATASET_ID)
# details

In [22]:
# dataset = dc_load_dataset(DATASET_ID)

Alternatively, download the dataset using curl.

In [23]:
# # Get download URL to download file
# ! RESPONSE=$(curl -X POST "https://datacollective.mozillafoundation.org/api/datasets/cmn2cy2su01iymm07xfr6ul2b/download" \
#   -H "Authorization: Bearer $(echo $MDC_API_KEY)" \
#   -H "Content-Type: application/json")

# # Extract download URL and download file
# ! DOWNLOAD_URL=$(echo $RESPONSE | jq -r '.downloadUrl')
# ! curl -C - -o "Common Voice Scripted Speech 25.0 - Dagbani.tar.gz" "$DOWNLOAD_URL"

In [24]:
# Mount Google Drive (if using Colab)
# from google.colab import drive
# drive.mount('/gdrive')

In [25]:
# On Kaggle
! pip install gdown -q
! gdown '1hgNt4Itm1aGonjjP78vjuRwy80iuEIJI' -O /tmp/Common\ Voice\ Scripted\ Speech\ 25.0\ -\ Dagbani.tar.gz --continue

Skipping already downloaded file /tmp/Common Voice Scripted Speech 25.0 - Dagbani.tar.gz


In [26]:
# Kaggle
dataset_file = "/tmp/Common Voice Scripted Speech 25.0 - Dagbani.tar.gz"
cv_dag_path = "/tmp/cv-corpus-25.0-2026-03-09"

if not os.path.exists(cv_dag_path):
    print(f"Extracting {dataset_file}...")
    with tarfile.open(dataset_file, "r:gz") as tar:
        tar.extractall(path='/tmp/')

In [27]:
# Load the dataset
cv_dag_test_df = pd.read_csv(f"{cv_dag_path}/dag/test.tsv", sep="\t")

# Rename columns for consistency
cv_dag_test_df = cv_dag_test_df.rename(columns={"path": "audio", "sentence": "transcription"})

# Convert audio paths to full paths
cv_dag_test_df["audio"] = cv_dag_test_df["audio"].apply(lambda x: os.path.join(cv_dag_path, "dag", "clips", x))

cv_dag_test_df.describe(include="all")

,client_id,audio,sentence_id,transcription,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
count,1464,1464,1464,1464,4,1464.000000,1464.000000,921,414,75,0.0,1464,0.0
unique,44,1464,1464,1464,3,NaN,NaN,3,3,5,NaN,1,NaN
top,7d09ac3de9453178258f460f5fb1286cd4a19909d169bd...,/tmp/cv-corpus-25.0-2026-03-09/dag/clips/commo...,38aea7d5b9ed9ae59ebc2e299760dca36272418b16eaed...,Bihi mi daa duɣiri kom.,general,NaN,NaN,thirties,female_feminine,Begginer,NaN,dag,NaN
freq,145,1,1,1,2,NaN,NaN,469,283,51,NaN,1464,NaN
mean,NaN,NaN,NaN,NaN,NaN,2.060109,0.079918,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,0.296609,0.273767,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,2.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,2.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,2.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,2.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# Convert audio paths to dict format for HF Dataset
cv_dag_test_df["audio"] = cv_dag_test_df["audio"].apply(lambda x: {'path': x})

# Convert to HF dataset
cv_dag_test = Dataset.from_pandas(cv_dag_test_df)

# Cast audio column to Audio type and resample to 16kHz
cv_dag_test = cv_dag_test.cast_column("audio", Audio(sampling_rate=16000))

In [29]:
# Calculate duration for each audio sample and add it as a new column
cv_dag_test = cv_dag_test.map(calculate_duration)

# Sort the dataset by duration
cv_dag_test = cv_dag_test.sort("duration", reverse=True)

Map:   0%|          | 0/1464 [00:00<?, ? examples/s]

In [30]:
# Listen to a sample audio
print(f"Original transcription: {cv_dag_test[0]['transcription']}")
AudioDisplay(cv_dag_test[0]["audio"]["array"], rate=cv_dag_test[0]["audio"]["sampling_rate"])

Original transcription: Bia maa nimbila daa tuya.


In [31]:
data["dag_Latn"].append({"Common_Voice_25_0": cv_dag_test})

# Text Normalization/Preprocessing

In [32]:
text_normalize("Hello, World! This is a test. 12345", "eng")

'hello world this is a test'

In [33]:
def remove_omni_tags(text):
    # Remove the <laugh>, <noise>, <hesitation>, and <unintelligible> tags
    tags_to_remove = ["<laugh>", "<noise>", "<hesitation>", "<unintelligible>"]
    for tag in tags_to_remove:
        text = text.replace(tag, "")
    return text

In [34]:
# Normalize all transcriptions in the datasets
for lang in data:
    for dataset_info in data[lang]:
        for dataset_name, dataset in dataset_info.items():
            print(f"Normalizing {dataset_name} for {lang}...")
            if dataset_name == "Omni_Corpus":
                # Remove Omni tags before normalization
                dataset = dataset.map(lambda x: {"normalized_transcription": remove_omni_tags(x["transcription"])})
                dataset = dataset.map(lambda x: {"normalized_transcription": text_normalize(x["normalized_transcription"], lang.split("_")[0])})
            else:
                dataset = dataset.map(lambda x: {"normalized_transcription": text_normalize(x["transcription"], lang.split("_")[0])})
            # Update the dataset in the data structure
            dataset_info[dataset_name] = dataset

for lang in long_data:
    for dataset_info in long_data[lang]:
        for dataset_name, dataset in dataset_info.items():
            print(f"Normalizing {dataset_name} for {lang}...")
            if dataset_name == "Omni_Corpus":
                # Remove Omni tags before normalization
                dataset = dataset.map(lambda x: {"normalized_transcription": remove_omni_tags(x["transcription"])})
                dataset = dataset.map(lambda x: {"normalized_transcription": text_normalize(x["normalized_transcription"], lang.split("_")[0])})
            else:
                dataset = dataset.map(lambda x: {"normalized_transcription": text_normalize(x["transcription"], lang.split("_")[0])})
            # Update the dataset in the data structure
            dataset_info[dataset_name] = dataset

Normalizing Omni_Corpus for fuq_Latn...


Map:   0%|          | 0/422 [00:00<?, ? examples/s]

Map:   0%|          | 0/422 [00:00<?, ? examples/s]

Normalizing Fleurs for fuv_Latn...
Normalizing Waxal for ewe_Latn...
Normalizing Waxal for dag_Latn...
Normalizing Common_Voice_25_0 for dag_Latn...


Map:   0%|          | 0/1464 [00:00<?, ? examples/s]

Normalizing Moore_Speech_Corpora for mos_Latn...


Map:   0%|          | 0/1012 [00:00<?, ? examples/s]

Normalizing Omni_Corpus for fuv_Latn...
Normalizing Omni_Corpus for snk_Latn...


In [35]:
ewe_latn_waxal_features['normalized_transcription'] = ewe_latn_waxal_features['transcription']
data["ewe_Latn"][0]['Waxal'] = data["ewe_Latn"][0]['Waxal'].cast(ewe_latn_waxal_features)

In [36]:
dag_latn_waxal_features['normalized_transcription'] = dag_latn_waxal_features['transcription']
data["dag_Latn"][0]['Waxal'] = data["dag_Latn"][0]['Waxal'].cast(dag_latn_waxal_features)

# Evaluation

In [37]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [70]:
def evaluate_asr_model(data, inference_fn, model_name="Model"):
    """
    Evaluate an ASR model on multiple datasets across languages.

    Args:
        data: Dictionary containing language -> datasets structure
        inference_fn: Function that takes (audio, sample_rate, lang) and returns prediction
        model_name: Name of the model for logging purposes

    Returns:
        results: Dictionary containing evaluation results
    """
    results = {}

    for lang in data:
        results[lang] = {}
        for dataset_info in data[lang]:
            for dataset_name, dataset in dataset_info.items():
                print(f"Evaluating {model_name} on {dataset_name} for {lang}...")

                predictions = []
                references = []
                original_references = []
                durations = []

                # for sample in tqdm(dataset.take(8)):
                for sample in tqdm(dataset):
                    audio = sample["audio"]["array"]
                    sample_rate = sample["audio"]["sampling_rate"]
                    duration = sample["duration"] if "duration" in sample else len(audio) / sample_rate
                    reference = sample["normalized_transcription"]
                    original = sample["transcription"]
                    pred = inference_fn(audio, sample_rate, lang)

                    # print(f"\nOriginal:\t{original}")
                    # print(f"Reference:\t{reference}")
                    # print(f"Prediction:\t{pred}\n")
                    predictions.append(pred)
                    references.append(reference)
                    original_references.append(original)
                    durations.append(duration)

                # Calculate WER and CER
                wer_score = wer_metric.compute(predictions=predictions, references=references)
                cer_score = cer_metric.compute(predictions=predictions, references=references)

                results[lang][dataset_name] = {
                    "wer": wer_score,
                    "cer": cer_score,
                    "predictions": predictions,
                    "references": references,
                    "originals": original_references,
                    "durations": durations
                }

                print(f"WER for {dataset_name} on {lang}: {wer_score}")
                print(f"CER for {dataset_name} on {lang}: {cer_score}\n")

    return results


def batch_evaluate_asr_model(data, batch_inference_fn, model_name="Model", batch_size=8):
    """
    Evaluate an ASR model on multiple datasets across languages using batch inference.

    Args:
        data: Dictionary containing language -> datasets structure
        batch_inference_fn: Function that takes (list of audios, list of sample_rates, lang) and returns list of predictions
        model_name: Name of the model for logging purposes
        batch_size: Number of samples to process in a batch

    Returns:
        results: Dictionary containing evaluation results
    """
    results = {}

    for lang in data:
        results[lang] = {}
        for dataset_info in data[lang]:
            for dataset_name, dataset in dataset_info.items():
                print(f"Evaluating {model_name} on {dataset_name} for {lang} with batch size {batch_size}...")

                references = []
                original_references = []
                durations = []
                predictions = []

                # for batch in tqdm(dataset.take(32).iter(batch_size)):
                for batch in tqdm(dataset.iter(batch_size)):
                    audios = [audio["array"] for audio in batch['audio']]
                    sample_rates = [audio["sampling_rate"] for audio in batch['audio']]
                    durations.extend(batch["duration"] if "duration" in batch else [len(audio) / sr for audio, sr in zip(audios, sample_rates)])
                    references.extend(batch["normalized_transcription"])
                    original_references.extend(batch["transcription"])
                    batch_predictions = batch_inference_fn(audios, sample_rates, lang)
                    # batch_start_idx = len(predictions)
                    predictions.extend(batch_predictions)

                    # for i in range(len(batch_predictions)):
                        # print(f"\nOriginal:\t{original_references[batch_start_idx + i]}")
                        # print(f"Reference:\t{references[batch_start_idx + i]}")
                        # print(f"Prediction:\t{batch_predictions[i]}\n")

                # Calculate WER and CER
                wer_score = wer_metric.compute(predictions=predictions, references=references)
                cer_score = cer_metric.compute(predictions=predictions, references=references)

                results[lang][dataset_name] = {
                    "wer": wer_score,
                    "cer": cer_score,
                    "predictions": predictions,
                    "references": references,
                    "originals": original_references,
                    "durations": durations
                }

                print(f"WER for {dataset_name} on {lang}: {wer_score}")
                print(f"CER for {dataset_name} on {lang}: {cer_score}\n")

    return results

In [39]:
# Create a NeMo-style manifest for each model-dataset combination
def create_manifest(results, model_name):
    for lang in results:
        for dataset_name in results[lang]:
            predictions = results[lang][dataset_name]["predictions"]
            references = results[lang][dataset_name]["references"]
            originals = results[lang][dataset_name]["originals"]
            durations = results[lang][dataset_name]["durations"]

            manifest = []
            for pred, ref, orig, dur in zip(predictions, references, originals, durations):
                manifest.append({
                    "audio_filepath": "",
                    "text": ref,
                    "original_text": orig,
                    "pred_text": pred,
                    "duration": dur
                })
            # Save manifest to JSONL file
            with open(f"{model_name}_{dataset_name}_{lang}_manifest.jsonl", "w") as f:
                for item in manifest:
                    json.dump(item, f)
                    f.write("\n")

In [40]:
# Store WER and CER scores in a CSV file for easier analysis
def results_to_df(results, model_name):
    rows = []
    for lang in results:
        for dataset_name in results[lang]:
            wer = results[lang][dataset_name]["wer"]
            cer = results[lang][dataset_name]["cer"]
            rows.append({
                "model": model_name,
                "language": lang,
                "dataset": dataset_name,
                "wer": wer,
                "cer": cer
            })
    df = pd.DataFrame(rows)
    return df

## Omnilingual ASR

In [41]:
# Print all supported languages
print(f"Total supported languages: {len(supported_langs)}")

for lang in set(list(data.keys()) + list(long_data.keys())):
    if lang in supported_langs:
        print(lang + " is supported by OmniASR!")
    else:
        print(lang + " is not supported by OmniASR!")

Total supported languages: 1672
fuv_Latn is supported by OmniASR!
snk_Latn is supported by OmniASR!
mos_Latn is supported by OmniASR!
ewe_Latn is supported by OmniASR!
dag_Latn is supported by OmniASR!
fuq_Latn is supported by OmniASR!


In [42]:
# Initialize pipeline
OMNI_MODEL_VARIANT = "omniASR_LLM_Unlimited_7B_v2"
omni_pipeline = ASRInferencePipeline(model_card=OMNI_MODEL_VARIANT)

def omni_asr_inference(audio, sample_rate, lang):
    if sample_rate != 16_000:
        print(f"OmniASR model expects audio to be sampled at 16kHz, but got {sample_rate}Hz.")

    try:
        pred = omni_pipeline.transcribe([{"waveform": audio, "sample_rate": sample_rate}], lang=[lang], batch_size=1)[0]
    except Exception as e:
        print(f"Error during inference: {e}")
        # OmniASR unlimited variants can sometimes throw an error. In such cases, we add silence to the end of the audio and try again.
        print("Adding 1 second of silence to the end of the audio and trying again...")
        silence = np.zeros(sample_rate, dtype=np.float32)  # 1 second of silence
        padded_waveform = np.concatenate([audio, silence])
        pred = omni_pipeline.transcribe([{"waveform": padded_waveform, "sample_rate": sample_rate}], lang=[lang], batch_size=1)[0]

    # OmniASR models can output the UNK token which gets decoded as " ⁇ ". Replace it with empty string for fair evaluation.
    # More info: https://github.com/facebookresearch/omnilingual-asr/issues/35
    if " ⁇ " in pred:
        print(f"Found UNK token in prediction: {pred}. Replacing it with empty string for evaluation.")
        pred = pred.replace(" ⁇ ", "")

    return pred

def omni_asr_batch_inference(audios, sample_rates, lang):
    omni_audio_data = [{"waveform": audio, "sample_rate": sr} for audio, sr in zip(audios, sample_rates)]
    preds = omni_pipeline.transcribe(omni_audio_data, lang=[lang]*len(audios), batch_size=len(audios))
    cleaned_preds = []
    for pred in preds:
        if " ⁇ " in pred:
            print(f"Found UNK token in prediction: {pred}. Replacing it with empty string for evaluation.")
            pred = pred.replace(" ⁇ ", "")
        cleaned_preds.append(pred)
    return cleaned_preds

Output()

In [43]:
# OmniASR eval on long data
omni_long_results = batch_evaluate_asr_model(long_data, omni_asr_batch_inference, model_name=OMNI_MODEL_VARIANT, batch_size=16)

Evaluating omniASR_LLM_Unlimited_7B_v2 on Omni_Corpus for fuv_Latn with batch size 16...


3it [02:42, 54.78s/it]

Found UNK token in prediction:  ⁇   ⁇ n ⁇ y ⁇ a ⁇ n ⁇ d ⁇ e ⁇ r ⁇ e ⁇   ⁇ t ⁇ o ⁇   ⁇ d ⁇ i ⁇   ⁇ w ⁇ a ⁇ r ⁇ i ⁇   ⁇ a ⁇ y ⁇ i ⁇ ' ⁇ a ⁇ i ⁇   ⁇ k ⁇ u ⁇ l ⁇ l ⁇ u ⁇ m ⁇   ⁇ d ⁇ e ⁇   ⁇ w ⁇ a ⁇ r ⁇ t ⁇ a ⁇   ⁇ f ⁇ u ⁇ h ⁇   ⁇ b ⁇ a ⁇   ⁇ y ⁇ a ⁇ n ⁇ d ⁇ e ⁇ r ⁇ e ⁇   ⁇ j ⁇ u ⁇ l ⁇ w ⁇ e ⁇   ⁇   ⁇ s ⁇ a ⁇ b ⁇ o ⁇ d ⁇ a ⁇   ⁇ y ⁇ a ⁇ n ⁇ d ⁇ e ⁇ r ⁇ e ⁇   ⁇ d ⁇ e ⁇ n ⁇   ⁇ w ⁇ e ⁇ w ⁇ a ⁇ r ⁇ a ⁇ i ⁇   ⁇ e ⁇ n ⁇ t ⁇ a ⁇ r ⁇ e ⁇   ⁇ d ⁇ e ⁇ d ⁇ a ⁇ i ⁇   ⁇ l ⁇ o ⁇ k ⁇ a ⁇ s ⁇ h ⁇ i ⁇   ⁇ y ⁇ a ⁇ n ⁇ d ⁇ e ⁇ r ⁇ e ⁇   ⁇ d ⁇ e ⁇  ⁇   ⁇ m ⁇ u ⁇ w ⁇ u ⁇ m ⁇   ⁇   ⁇ s ⁇ u ⁇ d ⁇ d ⁇ a ⁇   ⁇ h ⁇ o ⁇ r ⁇ e ⁇   ⁇ m ⁇ u ⁇ w ⁇ u ⁇ m ⁇   ⁇   ⁇ s ⁇ u ⁇ d ⁇ d ⁇ a ⁇   ⁇ h ⁇ o ⁇ r ⁇ e ⁇   ⁇ m ⁇ u ⁇ w ⁇ u ⁇ m ⁇   ⁇   ⁇ g ⁇ o ⁇   ⁇ a ⁇ l ⁇ l ⁇ a ⁇ h ⁇   ⁇ b ⁇ a ⁇ r ⁇ k ⁇ i ⁇ w ⁇ i ⁇ n ⁇ i ⁇ g ⁇ o ⁇   ⁇   ⁇ k ⁇ u ⁇ l ⁇ l ⁇ u ⁇ m ⁇   ⁇   ⁇ b ⁇ a ⁇   ⁇ j ⁇ u ⁇ l ⁇ d ⁇ e ⁇   ⁇   ⁇ m ⁇ i ⁇ n ⁇   ⁇ w ⁇ o ⁇ n ⁇   ⁇ n ⁇ a ⁇ n ⁇ a ⁇   ⁇ b ⁇ e ⁇ l ⁇ w ⁇ u ⁇ m ⁇   ⁇ m ⁇ a ⁇ g ⁇ o ⁇   ⁇ s ⁇ o ⁇ s ⁇ a ⁇ i ⁇   ⁇   ⁇ s ⁇

4it [03:28, 51.58s/it]

Found UNK token in prediction:  ⁇ w ⁇ o ⁇ n ⁇   ⁇ l ⁇ o ⁇ t ⁇ i ⁇ r ⁇ a ⁇   ⁇ j ⁇ u ⁇ d ⁇ e ⁇   ⁇ d ⁇ e ⁇ n ⁇   ⁇ e ⁇ n ⁇ t ⁇ a ⁇ r ⁇ e ⁇   ⁇ s ⁇ a ⁇ b ⁇ u ⁇ n ⁇ d ⁇ e ⁇   ⁇   ⁇ t ⁇ o ⁇   ⁇ t ⁇ a ⁇ t ⁇ a ⁇ b ⁇ o ⁇ l ⁇   ⁇ b ⁇ o ⁇   ⁇ d ⁇ e ⁇ w ⁇ o ⁇ n ⁇   ⁇ h ⁇ u ⁇ w ⁇ a ⁇ n ⁇ a ⁇   ⁇ a ⁇ l ⁇ ' ⁇ u ⁇ m ⁇ m ⁇ a ⁇ r ⁇ e ⁇   ⁇ h ⁇ a ⁇ r ⁇ o ⁇   ⁇ l ⁇ o ⁇ t ⁇ u ⁇ k ⁇ i ⁇   ⁇ s ⁇ u ⁇ t ⁇ u ⁇ r ⁇ a ⁇   ⁇ m ⁇ u ⁇ w ⁇ u ⁇ m ⁇   ⁇   ⁇ k ⁇ o ⁇ b ⁇ o ⁇   ⁇ w ⁇ u ⁇ w ⁇ w ⁇ u ⁇ m ⁇   ⁇ m ⁇ u ⁇ w ⁇ u ⁇ m ⁇   ⁇   ⁇ n ⁇ o ⁇   ⁇ w ⁇ u ⁇ n ⁇  ⁇   ⁇ w ⁇ i ⁇ t ⁇ i ⁇ r ⁇ i ⁇   ⁇ k ⁇ o ⁇   ⁇ b ⁇ o ⁇ r ⁇ n ⁇ o ⁇ t ⁇ o ⁇   ⁇   ⁇ k ⁇ a ⁇ y ⁇ a ⁇   ⁇ b ⁇ o ⁇ r ⁇ n ⁇ u ⁇ g ⁇ a ⁇ l ⁇   ⁇ m ⁇ u ⁇ w ⁇ u ⁇ m ⁇   ⁇   ⁇ n ⁇ a ⁇ n ⁇ g ⁇ u ⁇   ⁇ d ⁇ a ⁇ g ⁇ a ⁇   ⁇ j ⁇ o ⁇ n ⁇   ⁇ p ⁇ a ⁇ m ⁇ a ⁇ r ⁇ o ⁇ n ⁇   ⁇   ⁇ d ⁇ a ⁇ r ⁇ d ⁇ u ⁇ m ⁇ a ⁇ r ⁇ e ⁇   ⁇   ⁇ e ⁇ d ⁇ e ⁇   ⁇ i ⁇ r ⁇ i ⁇   ⁇ m ⁇ a ⁇ j ⁇ i ⁇   ⁇ e ⁇   ⁇ b ⁇ a ⁇ n ⁇ d ⁇ i ⁇ j ⁇ i ⁇   ⁇   ⁇ t ⁇ o ⁇   ⁇ h ⁇ a ⁇ n ⁇ j ⁇ u ⁇ m ⁇   ⁇ w ⁇ u ⁇ n ⁇   ⁇ h ⁇ u ⁇ t ⁇

5it [03:56, 47.22s/it]


Found UNK token in prediction: n ⁇   ⁇ m ⁇ a ⁇ r ⁇ e ⁇   ⁇ k ⁇ o ⁇ m ⁇ a ⁇ j ⁇ u ⁇ m ⁇   ⁇   ⁇ a ⁇ w ⁇ a ⁇ w ⁇ a ⁇ i ⁇   ⁇ n ⁇ a ⁇ f ⁇ i ⁇ r ⁇ k ⁇ i ⁇   ⁇ e ⁇   ⁇ m ⁇ a ⁇ a ⁇ j ⁇ u ⁇ m ⁇   ⁇   ⁇ a ⁇ y ⁇ i ⁇ i ⁇   ⁇   ⁇ t ⁇ o ⁇ h ⁇ a ⁇ r ⁇   ⁇ h ⁇ a ⁇ n ⁇ d ⁇ e ⁇  ⁇   ⁇ k ⁇ o ⁇ w ⁇ a ⁇   ⁇ n ⁇ o ⁇   ⁇ k ⁇ o ⁇   ⁇ n ⁇ a ⁇ w ⁇ i ⁇ r ⁇ t ⁇ a ⁇   ⁇ b ⁇ e ⁇ l ⁇ w ⁇ u ⁇ m ⁇   ⁇ h ⁇ u ⁇ d ⁇ u ⁇ k ⁇ i ⁇   ⁇ e ⁇   ⁇ m ⁇ a ⁇ a ⁇ j ⁇ u ⁇ m ⁇   ⁇   ⁇ t ⁇ o ⁇ h ⁇ a ⁇ r ⁇   ⁇ h ⁇ u ⁇ d ⁇ a ⁇ t ⁇ a ⁇   ⁇ e ⁇   ⁇ m ⁇ a ⁇ r ⁇ e ⁇   ⁇   ⁇ g ⁇  ⁇ o ⁇ w ⁇ a ⁇ i ⁇   ⁇ l ⁇ o ⁇ t ⁇ i ⁇ r ⁇ e ⁇   ⁇ e ⁇   ⁇ m ⁇ a ⁇ r ⁇ e ⁇   ⁇   ⁇ l ⁇ o ⁇ t ⁇ o ⁇ w ⁇ a ⁇   ⁇ e ⁇   ⁇ m ⁇ a ⁇ r ⁇ e ⁇   ⁇   ⁇   ⁇ a ⁇ ' ⁇ i ⁇ r ⁇ a ⁇   ⁇ e ⁇   ⁇ m ⁇ a ⁇ r ⁇ e ⁇   ⁇   ⁇   ⁇ t ⁇ o ⁇ h ⁇ . Replacing it with empty string for evaluation.
WER for Omni_Corpus on fuv_Latn: 0.5385342383366268
CER for Omni_Corpus on fuv_Latn: 0.22641411340634965

Evaluating omniASR_LLM_Unlimited_7B_v2 on Omni_Corpus for snk_Latn with batch size 16...


5it [04:41, 56.26s/it]

WER for Omni_Corpus on snk_Latn: 0.5462865188313974
CER for Omni_Corpus on snk_Latn: 0.15724662206138867



In [44]:
# OmniASR eval on short data
omni_short_results = batch_evaluate_asr_model(data, omni_asr_batch_inference, model_name=OMNI_MODEL_VARIANT, batch_size=32)

Evaluating omniASR_LLM_Unlimited_7B_v2 on Omni_Corpus for fuq_Latn with batch size 32...


14it [03:42, 15.86s/it]


WER for Omni_Corpus on fuq_Latn: 0.7547074562067747
CER for Omni_Corpus on fuq_Latn: 0.3085081402629931

Evaluating omniASR_LLM_Unlimited_7B_v2 on Fleurs for fuv_Latn with batch size 32...


1it [00:44, 44.20s/it]

Found UNK token in prediction:  ⁇   ⁇   ⁇ a ⁇ ' ⁇ a ⁇   ⁇ b ⁇ a ⁇ k ⁇ o ⁇   ⁇ f ⁇ a ⁇ r ⁇ i ⁇ j ⁇ a ⁇   ⁇ b ⁇ i ⁇ ' ⁇ a ⁇ t ⁇ i ⁇   ⁇ b ⁇ i ⁇   ⁇ f ⁇ a ⁇ s ⁇ i ⁇ t ⁇ i ⁇   ⁇ h ⁇ a ⁇ l ⁇ a ⁇   ⁇ s ⁇ a ⁇ r ⁇ i ⁇   ⁇ b ⁇ i ⁇   ⁇ w ⁇ a ⁇ d ⁇ i ⁇   ⁇ m ⁇ a ⁇ n ⁇   ⁇ g ⁇ a ⁇ m ⁇   ⁇ s ⁇ o ⁇ j ⁇ a ⁇   ⁇ b ⁇ u ⁇ r ⁇   ⁇ w ⁇ a ⁇ w ⁇ a ⁇ i ⁇   ⁇   ⁇   ⁇ 1800 ⁇   ⁇ d ⁇ i ⁇   ⁇ g ⁇ a ⁇ k ⁇ e ⁇ g ⁇ e ⁇   ⁇   ⁇ r ⁇ u ⁇ s ⁇ s ⁇ i ⁇ a ⁇   ⁇   ⁇ e ⁇ n ⁇   ⁇ n ⁇ a ⁇ s ⁇ t ⁇ i ⁇   ⁇ p ⁇ e ⁇ l ⁇ e ⁇ l ⁇   ⁇ m ⁇ a ⁇ n ⁇   ⁇ b ⁇ i ⁇   ⁇ w ⁇ a ⁇ d ⁇ i ⁇   ⁇ d ⁇ u ⁇ m ⁇ j ⁇ i ⁇   ⁇   ⁇ s ⁇ o ⁇ j ⁇ a ⁇   ⁇ b ⁇ e ⁇ l ⁇ e ⁇ r ⁇ u ⁇ s ⁇   ⁇   ⁇ e ⁇ n ⁇   ⁇ b ⁇ i ⁇ ' ⁇ a ⁇   ⁇ k ⁇ u ⁇ n ⁇   ⁇ b ⁇ a ⁇ w ⁇ o ⁇ d ⁇ i ⁇  ⁇   ⁇ n ⁇ a ⁇ s ⁇ t ⁇ i ⁇   ⁇ g ⁇ e ⁇ f ⁇ e ⁇ l ⁇ e ⁇   ⁇ t ⁇ u ⁇ g ⁇ a ⁇ l ⁇   ⁇ j ⁇ i ⁇   ⁇ b ⁇ u ⁇ r ⁇ o ⁇   ⁇   ⁇ p ⁇ o ⁇ l ⁇ a ⁇ n ⁇ d ⁇   ⁇   ⁇ w ⁇ a ⁇ d ⁇ u ⁇ k ⁇ o ⁇   ⁇ b ⁇ a ⁇ n ⁇ i ⁇ n ⁇ g ⁇   ⁇ m ⁇ a ⁇   ⁇ w ⁇ a ⁇ t ⁇ a ⁇   ⁇   ⁇ b ⁇ i ⁇   ⁇ f ⁇ i ⁇ l ⁇ t ⁇ i ⁇   ⁇ h ⁇ a ⁇ l ⁇ a ⁇   ⁇ j ⁇ 

...

WER for Waxal on dag_Latn: 0.6573922748834058
CER for Waxal on dag_Latn: 0.23956418357349654

Evaluating omniASR_LLM_Unlimited_7B_v2 on Common_Voice_25_0 for dag_Latn with batch size 32...


46it [01:08,  1.49s/it]


WER for Common_Voice_25_0 on dag_Latn: 0.27779702112919985
CER for Common_Voice_25_0 on dag_Latn: 0.07669964600163384

Evaluating omniASR_LLM_Unlimited_7B_v2 on Moore_Speech_Corpora for mos_Latn with batch size 32...


32it [01:19,  2.49s/it]

WER for Moore_Speech_Corpora on mos_Latn: 0.29511728099569173
CER for Moore_Speech_Corpora on mos_Latn: 0.10563895473034267



In [45]:
create_manifest(omni_short_results, OMNI_MODEL_VARIANT)
create_manifest(omni_long_results, OMNI_MODEL_VARIANT)

In [46]:
omni_short_df = results_to_df(omni_short_results, OMNI_MODEL_VARIANT)
omni_long_df = results_to_df(omni_long_results, OMNI_MODEL_VARIANT)

omni_df = pd.concat([omni_short_df, omni_long_df], ignore_index=True)
omni_df.to_csv("OmniASR_evaluation_results.csv", index=False)

In [47]:
# Free up GPU memory
del omni_pipeline
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated())

8519680


## Simba

In [48]:
# Load Simba-S for ASR
simba_pipeline = pipeline(
    "automatic-speech-recognition",
    model="UBC-NLP/Simba-S",
    dtype=torch.bfloat16,
)

def simba_asr_inference(audio, sample_rate, lang):
    # Simba-S model expects 16kHz audio
    if sample_rate != 16_000:
        raise ValueError(f"Simba-S ASR model expects audio to be sampled at 16kHz, but got {sample_rate}Hz")

    result = simba_pipeline({
        "array": audio,
        "sampling_rate": sample_rate
    })
    return result["text"]

def simba_batch_inference(audios, sample_rates, lang):
    data = [{'array': array, 'sampling_rate': sampling_rate} for array, sampling_rate in zip(audios, sample_rates)]

    results = simba_pipeline(data, batch_size=len(data))

    return [result['text'] for result in results]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.01G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/31.2M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [49]:
# This silences all warnings from the transformers library
logging.set_verbosity_error()

In [50]:
# Only evaluate on Ewe and Moore
simba_data = {lang: data[lang] for lang in data if lang in ["ewe_Latn", "mos_Latn"]}

#simba_results = evaluate_asr_model(simba_data, simba_asr_inference, model_name="Simba-S")
simba_results = batch_evaluate_asr_model(simba_data, simba_batch_inference, model_name="Simba-S", batch_size=64)

Evaluating Simba-S on Waxal for ewe_Latn with batch size 64...


60it [19:38, 19.65s/it]


WER for Waxal on ewe_Latn: 1.1310676529413632
CER for Waxal on ewe_Latn: 0.5111763014486403

Evaluating Simba-S on Moore_Speech_Corpora for mos_Latn with batch size 64...


16it [01:50,  6.88s/it]

WER for Moore_Speech_Corpora on mos_Latn: 1.1457635232168502
CER for Moore_Speech_Corpora on mos_Latn: 0.7504462587422819



In [51]:
create_manifest(simba_results, "Simba-S")

In [52]:
simba_df = results_to_df(simba_results, "Simba-S")
simba_df.to_csv("Simba_evaluation_results.csv", index=False)

In [53]:
# Free up GPU memory
del simba_pipeline
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated())

8519680


## MMS

In [54]:
# # Restore the original audio after processing
# omni_fuv_test = omni_fuv_test.cast_column("audio", Audio(decode=False))
# omni_fuq_test = omni_fuq_test.cast_column("audio", Audio(decode=False))

# omni_fuv_test = omni_fuv_test.map(restore_original_audio)
# omni_fuq_test = omni_fuq_test.map(restore_original_audio)

# omni_fuv_test = omni_fuv_test.cast_column("audio", Audio(decode=True, sampling_rate=16000))
# omni_fuq_test = omni_fuq_test.cast_column("audio", Audio(decode=True))

# long_data["fuv_Latn"][0]['Omni_Corpus'] = omni_fuv_test
# data["fuq_Latn"][0]['Omni_Corpus'] = omni_fuq_test

In [55]:
# # Listen to the fixed audio samples to verify they are correct
# for sample in long_data["fuv_Latn"][0]['Omni_Corpus']:
#     if sample['idx'] in fuv_idx:
#         print(f"Sample {sample['idx']}\nOriginal transcription: {sample['transcription']}\nDuration: {sample['duration']} seconds")
#         display(AudioDisplay(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]))
# for sample in data["fuq_Latn"][0]['Omni_Corpus']:
#     if sample['idx'] in fuq_idx:
#         print(f"Sample {sample['idx']}\nOriginal transcription: {sample['transcription']}\nDuration: {sample['duration']} seconds")
#         display(AudioDisplay(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]))

In [66]:
# load model
model_id = "facebook/mms-1b-all"

mms_processor = AutoProcessor.from_pretrained(model_id)
mms_model = Wav2Vec2ForCTC.from_pretrained(model_id).to('cuda')

In [67]:
for lang in set(list(data.keys()) + list(long_data.keys())):
    if lang.split("_")[0] not in mms_processor.tokenizer.vocab.keys():
        print(f"{lang} is not supported by MMS model.")
    else:
        print(f"{lang} is supported by MMS model.")

fuv_Latn is not supported by MMS model.
snk_Latn is not supported by MMS model.
mos_Latn is supported by MMS model.
ewe_Latn is supported by MMS model.
dag_Latn is not supported by MMS model.
fuq_Latn is not supported by MMS model.


In [68]:
def mms_asr_inference(audio, sample_rate, lang):
    # MMS model expects 16kHz audio
    if sample_rate != 16_000:
        raise ValueError(f"MMS ASR model expects audio to be sampled at 16kHz, but got {sample_rate}Hz")

    # Load language adapter if needed
    tgt_lang = lang.split("_")[0]
    if mms_model.target_lang != tgt_lang or mms_processor.tokenizer.target_lang != tgt_lang:
        print(f"Loading language adapter for {tgt_lang}...")
        mms_model.load_adapter(tgt_lang)
        mms_processor.tokenizer.set_target_lang(tgt_lang)

    # Preprocess audio
    inputs = mms_processor(audio, sampling_rate=sample_rate, return_tensors="pt").to('cuda')

    # Run inference
    with torch.no_grad():
        outputs = mms_model(**inputs).logits
    ids = torch.argmax(outputs, dim=-1)[0]

    pred = mms_processor.decode(ids)

    return pred

In [71]:
# Only evaluate on Ewe and Moore
mms_data = {lang: data[lang] for lang in data if lang in ["ewe_Latn", "mos_Latn"]}

mms_results = evaluate_asr_model(mms_data, mms_asr_inference, model_name="MMS")

Evaluating MMS on Waxal for ewe_Latn...


3782it [08:29,  7.42it/s]


WER for Waxal on ewe_Latn: 0.50352392139433
CER for Waxal on ewe_Latn: 0.1536818468110743

Evaluating MMS on Moore_Speech_Corpora for mos_Latn...


  0%|          | 0/1012 [00:00<?, ?it/s]

Loading language adapter for mos...


adapter.mos.safetensors:   0%|          | 0.00/8.84M [00:00<?, ?B/s]

100%|██████████| 1012/1012 [00:56<00:00, 18.06it/s]


WER for Moore_Speech_Corpora on mos_Latn: 0.43752991862134993
CER for Moore_Speech_Corpora on mos_Latn: 0.14619729025839112



In [72]:
create_manifest(mms_results, "MMS")

In [73]:
mms_df = results_to_df(mms_results, "MMS")
mms_df.to_csv("MMS_evaluation_results.csv", index=False)

# Process results

In [74]:
# Combine all results into a single DataFrame
all_results_df = pd.concat([omni_df, mms_df, simba_df], ignore_index=True)
all_results_df.to_csv("ASR_evaluation_results.csv", index=False)

all_results_df

,model,language,dataset,wer,cer
0,omniASR_LLM_Unlimited_7B_v2,fuq_Latn,Omni_Corpus,0.754707,0.308508
1,omniASR_LLM_Unlimited_7B_v2,fuv_Latn,Fleurs,0.480730,0.148944
2,omniASR_LLM_Unlimited_7B_v2,ewe_Latn,Waxal,0.413958,0.129686
3,omniASR_LLM_Unlimited_7B_v2,dag_Latn,Waxal,0.657392,0.239564
4,omniASR_LLM_Unlimited_7B_v2,dag_Latn,Common_Voice_25_0,0.277797,0.076700
5,omniASR_LLM_Unlimited_7B_v2,mos_Latn,Moore_Speech_Corpora,0.295117,0.105639
6,omniASR_LLM_Unlimited_7B_v2,fuv_Latn,Omni_Corpus,0.538534,0.226414
7,omniASR_LLM_Unlimited_7B_v2,snk_Latn,Omni_Corpus,0.546287,0.157247
8,MMS,ewe_Latn,Waxal,0.503524,0.153682
9,MMS,mos_Latn,Moore_Speech_Corpora,0.437530,0.146197
